# TurboQuant x Refusal Eval (Kaggle)

This notebook runs baseline vs TurboQuant-style KV cache configurations and reports refusal rate + KL drift.

## 1) Clone and install

If you already attached the repo as a Kaggle Dataset, update paths accordingly.

In [ ]:
# !git clone https://github.com/<your-username>/turboquant-heretic-kv-bench.git
# %cd turboquant-heretic-kv-bench
# !pip install -e .

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from tqhk.benchmark import BenchmarkConfig, RunConfig, run_benchmark
from tqhk.cache import CacheConfig

In [ ]:
cfg = BenchmarkConfig(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    device='cuda',
    load_in_4bit=True,
    harmless_split='test[:100]',
    harmful_split='test[:100]',
    batch_size=4,
    max_new_tokens=64,
    filler_repetitions=24,
    output_csv='results/benchmark_results.csv',
    output_json='results/benchmark_results.json',
)

runs = [
    RunConfig(
        name='baseline_fp16_cache',
        use_turboquant_cache=False,
        cache_config=CacheConfig(),
    ),
    RunConfig(
        name='tq_k8_v4_rw128',
        use_turboquant_cache=True,
        cache_config=CacheConfig(key_bits=8, value_bits=4, residual_mode='fixed', residual_window=128),
    ),
    RunConfig(
        name='tq_k6_v4_rw128',
        use_turboquant_cache=True,
        cache_config=CacheConfig(key_bits=6, value_bits=4, residual_mode='fixed', residual_window=128),
    ),
    RunConfig(
        name='tq_k8_v4_dynamic_rw',
        use_turboquant_cache=True,
        cache_config=CacheConfig(
            key_bits=8,
            value_bits=4,
            residual_mode='dynamic',
            dynamic_fraction=0.05,
            dynamic_min=32,
            dynamic_max=256,
        ),
    ),
]

rows = run_benchmark(cfg=cfg, run_configs=runs)
rows

In [ ]:
import pandas as pd
df = pd.read_csv('results/benchmark_results.csv')
display(df[['run_name', 'refusal_rate', 'avg_kl_to_baseline', 'avg_latency_sec']])